### SARIMA Model

In [ ]:
# Demand forecast
import pandas as pd
import numpy as np
from babel.dates import parse_date
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

################### Load Data #####################################
AIL = pd.read_csv('ActualForecastData.csv', parse_dates=['date_he'])
AIL_indexed = AIL.set_index('date_he')
daily_sum = AIL_indexed['actual_ail'].resample('D').sum()
daily_count = AIL_indexed['actual_ail'].resample('D').count()

# Create time series.
# Keep weeks within ±1 hour of full (handles DST: 167 and 169 are both valid)
# Keep only days with full observations (handles DST: 23 or 25 hours)
AIL = pd.read_csv('ActualForecastData.csv', parse_dates=['date_he'])
AIL_indexed = AIL.set_index('date_he')
daily_sum = AIL_indexed['actual_ail'].resample('D').sum()
daily_count = AIL_indexed['actual_ail'].resample('D').count()

full_day = daily_count.mode()[0]
demand_data = daily_sum[(daily_count >= full_day - 1) & (daily_count <= full_day + 1)]
print(demand_data.info())

stl = STL(demand_data, period=7, seasonal=13)
result = stl.fit()
result.plot()
plt.tight_layout()
plt.show()



#### Stationary Check (ADF test)
ADF tests the null hypothesis that there is the existence of a unit root in the time series sample.
The presence of a unit root implies that the time series is non-stationary, that is,
its statistical properties such as mean and variance change over time and the series
follows a stochastic trend rather than reverting to a fixed mean.

In [ ]:
def adftest(dataset):
    try:
        result = adfuller(dataset, autolag='AIC')
        print(f'ADF Statistic: {result[0]:.4f}')
        print(f'p-value: {result[1]:.4f}')
        print(f'Lags used: {result[2]}')
        print(f'Critical values: {result[4]}')
        if result[1] < 0.05:
            print('Time Series is Stationary')
        else:
            print('Fail to reject null hypothesis. Time Series is Non-Stationary')
    except Exception as e:
        print(f'Invalid dataset: {e}')

adftest(demand_data)

#### Differencing
Differencing is a method used to transform a non-stationary time series by calculating
the difference between consecutive observations.

$$y'_t = y_t - y_{t-1}$$

It stabilizes the mean of a time series by removing changes in level and trend,
making it more suitable for modelling.

In [ ]:
def adftest(dataset):
    try:
        result = adfuller(dataset, autolag='AIC')
        return result[1] < 0.05  # return True if stationary
    except Exception as e:
        print(f'Invalid dataset: {e}')
        return None

def difference(dataset, interval=1):
    if adftest(dataset):  # if stationary, skip
        print('Time Series is already stationary, no differencing needed')
        return dataset
    diff = []
    for i in range(interval, len(dataset)):
        diff.append(dataset.iloc[i] - dataset.iloc[i - interval])
    return pd.Series(diff)

difference(demand_data).plot()
plt.show()

#### Model Parameters
**Autocorrelation function (ACF):** Measures the correlation between the past values (lagged values) and the current value, including indirect effects. We will use the ACF to find Moving Average parameter q.

**Partial Autocorrelation function (PACF):** Measures the direct correlation between the k-th lagged value and the current value, removing indirect effects. We will use the PACF to find the Autoregressive parameter p.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 8))

plot_acf(difference(demand_data,1), ax=ax[0], lags=60)   # ~2 months of days
plot_pacf(difference(demand_data,1), ax=ax[1], lags=60)

plt.tight_layout()
plt.show()

### SARIMAX Model

In [ ]:
import itertools
import warnings
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt


def sarima_grid_search(train, test, m=7, p=range(3), q=range(3), P=range(2), Q=range(2)):
    """
    Grid search SARIMA orders, fixing d=1 and D=1 (known from the trend/seasonal
    decomposition). Models are ranked by forecast RMSE on the test set, with
    coefficient significance reported as a diagnostic rather than a sort key.
    """
    rows = []

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for pp, qq, PP, QQ in itertools.product(p, q, P, Q):
            order = (pp, 1, qq)
            seasonal_order = (PP, 1, QQ, m)
            try:
                fit = SARIMAX(train,
                              order=order,
                              seasonal_order=seasonal_order).fit(disp=False, maxiter=100)

                rmse = np.sqrt(mean_squared_error(test, fit.forecast(len(test))))
                pvals = fit.pvalues.drop('sigma2', errors='ignore')

                rows.append({
                    'order': order,
                    'seasonal_order': seasonal_order,
                    'RMSE': rmse,
                    'AIC': fit.aic,
                    'all_significant': bool((pvals < 0.05).all()),
                })
            except Exception:
                continue

    return pd.DataFrame(rows).sort_values('RMSE').reset_index(drop=True)


# Split (90/10) and set daily frequency
split = int(len(demand_data) * 0.90)
train = demand_data[:split].asfreq('D')
test = demand_data[split:].asfreq('D')

# Search
results = sarima_grid_search(train, test)
print(results.head(10))

# Fit best model
best = results.iloc[0]
fit = SARIMAX(train,
              order=best['order'],
              seasonal_order=best['seasonal_order']).fit(disp=False, maxiter=100)
print(fit.summary())

forecast = fit.forecast(len(test))
forecast.index = test.index

# Plot last 90 days of train + forecast vs actual
plt.figure(figsize=(15, 6))
plt.plot(train.index[-90:], train.values[-90:], 'b-', alpha=0.6, label='Train')
plt.plot(test.index, test.values, 'g-', lw=2, label='Actual')
plt.plot(forecast.index, forecast.values, 'r--', lw=2, label='Forecast')
plt.axvline(train.index[-1], color='black', ls=':', lw=2)
plt.legend(); plt.title('Split Forecast'); plt.grid(True); plt.show()

print(f"\nMAE:  {mean_absolute_error(test, forecast):,.0f} MWh")
print(f"RMSE: {np.sqrt(mean_squared_error(test, forecast)):,.0f} MWh")


### Conclusion
This SARIMAX model can only capture one seasonal period at a time, that is, it cannot simultaneously model the daily, weekly, and annual cycles present in the AIL data.
Not good for a long term horizon